# 03 Embeddings

This notebook builds quarter-specific GraphSAGE and Node2Vec embeddings and exports pooled datasets for later regression experiments in a separate notebook.


In [1]:
import sys
import os
from pathlib import Path

%matplotlib inline

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.embeddings import GNNConfig, Node2VecConfig
from src.models.embedding_pipeline import build_pooled_dataset

PROJECT_ROOT = Path().resolve().parents[1]
DATA_PATH = PROJECT_ROOT / 'datasets'
OUTPUT_ROOT = PROJECT_ROOT / 'src' / 'data'

pd.set_option("display.max_columns", 200)

## Configuration

In [2]:
cfg_graphsage = GNNConfig(
    hidden_dims=(256, 64),
    dropout=0.3,
    lr=0.01,
    epochs=100,
    aggregation="mean",
    device="cpu",
)

cfg_node2vec = Node2VecConfig(
    embedding_dim=64,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    batch_size=128,
    lr=0.01,
    epochs=100,
    device="cpu",
)

TARGET_COL = "log_systemic_risk_label"
INCLUDE_RAW_FEATURES = False

OUTPUT_DATASET_GRAPHSAGE = PROJECT_ROOT / "src" / "data" / "embeddings" / "graphsage_srisk_dataset.parquet"
OUTPUT_DATASET_NODE2VEC = PROJECT_ROOT / "src" / "data" / "embeddings" / "node2vec_srisk_dataset.parquet"

OUTPUT_DATASET_GRAPHSAGE, OUTPUT_DATASET_NODE2VEC


(WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/graphsage_srisk_dataset.parquet'),
 WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/node2vec_srisk_dataset.parquet'))

# Build Embedding Dataset

## GraphSAGE

For each quarter, GraphSAGE is trained on that quarter's graph only. The resulting embeddings are merged with the regression target `systemic_risk_label` and optionally with the raw node features.


In [3]:
pooled_df_graphsage = build_pooled_dataset(
    config=cfg_graphsage,
    years=range(2016, 2024),
    quarters=(1, 2, 3, 4),
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    output_path=OUTPUT_DATASET_GRAPHSAGE,
)

pooled_df_graphsage.shape

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


(145536, 69)

In [4]:
pooled_df_graphsage.head()

,bank_id,year,quarter,period,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,emb_32,emb_33,emb_34,emb_35,emb_36,emb_37,emb_38,emb_39,emb_40,emb_41,emb_42,emb_43,emb_44,emb_45,emb_46,emb_47,emb_48,emb_49,emb_50,emb_51,emb_52,emb_53,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63,log_systemic_risk_label
0,0,2016,1,2016Q1,0.035766,-1.120199,-1.405256,0.531244,2.430931,-0.641421,0.435702,5.556161,1.537100,1.826005,-0.864338,0.792052,-1.084507,-2.031601,1.359191,-0.850630,-0.342492,-1.088447,0.830222,0.001940,-0.885173,-1.611640,3.404722,1.177111,3.747552,-1.415333,1.044253,-0.871079,0.007087,0.862662,-2.793181,-6.586875,2.726410,0.730440,-0.364956,1.211423,0.769860,0.561812,-1.515466,2.989322,-4.269004,-0.298547,0.360449,-1.661506,0.463563,-1.329896,4.285762,-1.860508,-2.706345,-1.433310,0.534767,1.250288,-1.278694,-0.892721,-0.551547,-1.642586,-1.037910,2.867615,0.834992,-0.384309,-2.166135,1.333163,-2.148995,-1.116616,5.375278
1,1,2016,1,2016Q1,-0.111444,-1.292578,-1.578537,0.712835,2.716421,-0.774950,0.473357,5.228150,1.734732,2.122643,-0.873176,0.772484,-1.180336,-2.239998,1.389954,-0.938604,-0.325494,-1.180358,0.906852,0.095699,-1.010154,-1.848304,3.795864,1.177621,4.121108,-1.510794,1.114520,-0.583556,0.068925,0.983031,-2.853094,-6.477312,2.779228,0.695812,-0.577550,1.402118,0.920933,0.806403,-1.582042,2.844667,-4.196001,-0.248212,0.358239,-1.853878,0.488340,-1.477319,4.758703,-1.991323,-2.825448,-1.519093,0.531993,1.255490,-1.189356,-0.943070,-0.466429,-1.971109,-0.995330,3.218037,0.944370,-0.197394,-2.199000,1.608572,-1.947526,-1.255390,3.044522
2,2,2016,1,2016Q1,-0.045721,-0.811294,-0.848657,0.233694,1.560560,-0.669129,0.371992,4.118658,0.935623,0.769201,-0.703971,0.765063,-0.932898,-1.082494,1.250732,-0.464914,-0.651233,-0.785092,0.684807,0.039259,-0.355383,-0.938082,1.865944,0.568310,2.202739,-0.946785,0.354064,-0.788106,0.072366,0.284203,-1.529513,-5.332948,1.510524,0.594258,0.254182,0.586157,0.573817,-0.097164,-0.690485,2.204798,-3.186240,-0.688236,0.508843,-0.998017,0.258288,-0.900554,2.406981,-0.781005,-1.684868,-1.138739,0.584927,0.741197,-0.952480,-0.500436,-0.059003,-0.693081,-0.672408,1.823604,0.646102,-0.945228,-1.182795,0.453159,-2.088746,-0.435389,4.564348
3,3,2016,1,2016Q1,-0.035446,-1.296174,-1.576155,0.629433,2.752239,-0.790371,0.500326,5.851398,1.739078,2.021927,-0.966889,0.872328,-1.250570,-2.218336,1.548155,-0.948139,-0.447663,-1.214280,0.939976,0.075916,-0.977135,-1.838970,3.761107,1.206520,4.152612,-1.573498,1.115015,-0.832741,0.062006,0.915623,-2.957728,-7.181789,2.889004,0.799425,-0.413710,1.351343,0.890332,0.633057,-1.607665,3.194549,-4.592011,-0.404942,0.447049,-1.857635,0.499816,-1.494112,4.739858,-1.973511,-2.945115,-1.621653,0.629718,1.324536,-1.350554,-0.969349,-0.504582,-1.843797,-1.093854,3.229839,0.962848,-0.461181,-2.292296,1.487397,-2.330630,-1.203686,3.637586
4,4,2016,1,2016Q1,0.020626,-0.630699,-0.716375,0.208640,1.259513,-0.452709,0.279435,3.382903,0.748645,0.745667,-0.531184,0.562106,-0.691333,-0.983888,0.922673,-0.412040,-0.391580,-0.635401,0.536530,-0.023324,-0.345909,-0.754515,1.626762,0.554182,1.860062,-0.777208,0.380332,-0.645556,0.034264,0.309630,-1.383784,-4.154881,1.350716,0.468745,0.092131,0.529435,0.445574,0.035340,-0.662725,1.768996,-2.552841,-0.387883,0.309174,-0.838737,0.222972,-0.712873,2.073137,-0.773507,-1.419743,-0.856368,0.409999,0.651676,-0.765008,-0.426723,-0.164675,-0.656234,-0.576383,1.481670,0.492360,-0.565108,-1.063002,0.461504,-1.537214,-0.420406,3.367296


In [5]:
embedding_cols = [c for c in pooled_df_graphsage.columns if c.startswith("emb_")]
meta_cols = ["bank_id", "year", "quarter", "period", TARGET_COL]
meta_cols + embedding_cols[:5], len(embedding_cols)

(['bank_id',
  'year',
  'quarter',
  'period',
  'log_systemic_risk_label',
  'emb_0',
  'emb_1',
  'emb_2',
  'emb_3',
  'emb_4'],
 64)

In [6]:
pooled_df_graphsage.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

count      mean       max
year quarter                           
2016 1         4548  0.719746  5.375278
     2         4548  0.714352  4.727388
     3         4548  0.711462  4.189655
     4         4548  0.709951  3.367296
2017 1         4548  0.710255  3.526361
     2         4548  0.708800  3.737670
     3         4548  0.708698  3.555348
     4         4548  0.709404  3.465736
2018 1         4548  0.713800  3.713572
     2         4548  0.711474  3.637586
     3         4548  0.709318  3.637586
     4         4548  0.709583  3.401197

## Node2Vec

Node2Vec is trained on quarter-specific graph structure using random walks. The resulting embeddings are merged with the same downstream target for comparison against GraphSAGE.


In [7]:
pooled_df_node2vec = build_pooled_dataset(
    config=cfg_node2vec,
    years=range(2016, 2024),
    quarters=(1, 2, 3, 4),
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    output_path=OUTPUT_DATASET_NODE2VEC,
)

pooled_df_node2vec.shape


Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


(145536, 69)

In [8]:
pooled_df_node2vec.head()


,bank_id,year,quarter,period,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,emb_32,emb_33,emb_34,emb_35,emb_36,emb_37,emb_38,emb_39,emb_40,emb_41,emb_42,emb_43,emb_44,emb_45,emb_46,emb_47,emb_48,emb_49,emb_50,emb_51,emb_52,emb_53,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63,log_systemic_risk_label
0,0,2016,1,2016Q1,-0.553849,5.615181,-0.829854,2.841327,6.241623,1.965094,1.095165,0.316519,1.844672,-0.640510,-0.261085,0.064869,0.207421,1.637971,0.020891,-6.832967,-0.611003,-0.268902,5.825840,0.437195,-0.928425,1.389319,1.895392,-0.430654,-1.126530,-2.170848,-1.605871,-0.392710,-3.486688,-0.116934,-0.996246,1.181252,1.098326,-0.442363,2.280025,3.759004,-1.235928,-0.747886,1.118131,0.383464,1.781294,-0.229850,-2.187715,-2.445372,0.006368,-0.953537,0.213673,-0.996393,1.342621,0.020873,0.130629,1.666479,0.475700,3.097912,-0.618144,-4.202543,-0.733614,-0.240180,0.228931,3.051112,0.068276,-0.425772,-0.401855,0.421430,5.375278
1,1,2016,1,2016Q1,-0.797719,1.932361,0.080392,0.960906,4.656557,3.694211,-0.097560,0.263413,-0.152135,-0.178089,0.395586,-0.273306,-0.686823,1.420885,-0.427397,-3.200749,-0.088795,-0.306455,0.646762,-0.034086,-2.511172,0.234385,0.742288,-0.548292,-0.569280,-1.792224,-0.083251,-0.058683,-2.242608,0.022856,1.112292,0.402853,0.209905,-0.142439,0.736875,2.268709,-0.211184,0.783577,1.850233,-0.876065,1.287504,-0.545006,-1.386331,-2.319916,0.715051,-0.435197,-0.241065,-0.232902,-0.079607,0.074605,0.125379,0.686234,0.388715,1.071536,0.369659,-3.312312,-0.374080,0.222762,0.360410,3.753986,1.219777,0.124458,-0.081886,0.805266,3.044522
2,2,2016,1,2016Q1,-0.179833,2.326897,-0.556318,1.979616,2.866484,3.564239,0.050456,-0.320930,1.913152,-0.127082,-0.079816,-0.267414,-1.275355,1.664748,0.409365,-3.277638,-0.224202,-0.242817,2.741358,0.886062,-3.799477,0.811693,0.464945,-0.266862,0.142835,-0.119510,0.762158,0.500672,-1.477568,0.433890,-0.217441,1.545162,0.027926,-0.120705,1.091951,3.920874,-0.568608,-0.119623,0.488956,0.033841,1.086423,-1.365611,-1.266337,-2.511700,-0.197945,0.766624,-0.066952,-0.255388,-0.569525,0.802112,0.754129,0.508444,0.680993,3.505735,-1.065298,-1.336148,0.122361,-0.389166,-0.324132,3.656529,2.139058,-0.347379,-0.119427,0.597290,4.564348
3,3,2016,1,2016Q1,0.090867,1.465762,-0.128942,0.388533,2.275106,2.250386,-0.330312,-0.065772,-0.811652,-0.257857,-0.356740,-0.132260,-0.201364,0.675034,-0.325537,-2.044655,0.174295,-0.126472,1.863841,-0.419519,-2.782714,0.544116,1.826606,0.417839,-1.458948,-2.102330,-0.625584,-0.013632,-1.285623,0.130958,-0.383557,0.170555,0.044610,-0.237384,0.657121,3.160835,-0.770476,0.433573,2.073382,-0.599110,2.893120,-0.935522,-1.258479,-2.189204,-1.372601,-0.150889,-0.439714,-0.650771,1.112323,0.336070,-0.356371,-0.113138,-0.452844,3.401232,-0.597887,-4.652353,0.516224,0.321778,0.090497,1.678271,1.133551,0.231316,1.276898,0.928467,3.637586
4,4,2016,1,2016Q1,-0.746193,3.292065,-0.689761,0.539457,1.766822,4.441014,0.320599,0.298577,1.848384,-0.100728,0.172498,0.537931,-0.084379,0.405936,0.084517,-2.916117,-0.050622,0.273395,0.555646,0.344840,-2.293468,-0.980512,0.869555,-0.354306,-0.422874,-0.215213,-0.065090,-0.027158,-1.647122,0.049334,-0.156873,0.189994,0.091761,-0.286465,-0.041378,1.430581,-0.319035,0.866498,0.902060,0.331508,2.366619,-0.660543,-0.910655,-2.100700,0.097224,-1.075397,0.093610,0.272704,1.398593,-0.882818,0.247420,-0.358696,-0.412139,1.369683,0.308428,-1.989740,-0.811304,-0.279265,-0.896185,1.495422,2.666616,0.224162,0.055714,1.134122,3.367296


In [9]:
embedding_cols = [c for c in pooled_df_node2vec.columns if c.startswith("emb_")]
meta_cols = ["bank_id", "year", "quarter", "period", TARGET_COL]
meta_cols + embedding_cols[:5], len(embedding_cols)


(['bank_id',
  'year',
  'quarter',
  'period',
  'log_systemic_risk_label',
  'emb_0',
  'emb_1',
  'emb_2',
  'emb_3',
  'emb_4'],
 64)

In [10]:
pooled_df_node2vec.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)


count      mean       max
year quarter                           
2016 1         4548  0.719746  5.375278
     2         4548  0.714352  4.727388
     3         4548  0.711462  4.189655
     4         4548  0.709951  3.367296
2017 1         4548  0.710255  3.526361
     2         4548  0.708800  3.737670
     3         4548  0.708698  3.555348
     4         4548  0.709404  3.465736
2018 1         4548  0.713800  3.713572
     2         4548  0.711474  3.637586
     3         4548  0.709318  3.637586
     4         4548  0.709583  3.401197

# Exported Datasets

The saved datasets contain:
- metadata: `bank_id`, `year`, `quarter`, `period`
- target: `systemic_risk_label`
- embeddings: `emb_0`, `emb_1`, ...
- optional raw features from the node table

Files written by this notebook:
- `outputs/embeddings/graphsage_srisk_dataset.parquet`
- `outputs/embeddings/node2vec_srisk_dataset.parquet`


## Next Step

Use the exported parquet files in a separate notebook for classical machine-learning regression or model comparison.
